# FFM48k_PipelineFigure_Arrays — arrays for the Chapter 3 pipeline figure (`fig:pipeline`)

**Evaluation only. Nothing is trained, and no checkpoint or result file is changed.** The notebook reads FFM48k_measim (the frozen autoencoder of FFM_3_8), the 48 kHz data and FFM_3_8's out-of-fold predictions (`mesh2z_48k_v1/dev_oof.npz`). It saves the arrays for one subject, so the figure panels can be drawn locally.

**Rules (fixed before running)**

- **Subject:** SONICOM P0010, a development subject. It is the subject of `fig:sonicom-mesh` and `fig:ear-crop`. Its geometry prediction is FFM_3_8's out-of-fold prediction, so the subject was not in that fold's training data. No locked test subject is read.
- **Reproduction gate:** two values recorded in `dev_oof.npz` must be reproduced within 0.001 dB before anything is saved. They are P0010's LSD (bins 1–85, elevation ≥ −45°, mean of both ears) for (a) the decoded geometry prediction and (b) the training mean of its fold. This checks the row order (left / mirrored right), the FiLM row and the fold.
- **Saved:** `HRTF_Checkpoints/pipeline_fig/pipeline_P0010.npz`, left ear in its own frame: measured HRTF, AE reconstruction, latent z, predicted latent ẑ (geometry), decoded prediction, training mean of the fold, frequencies and the gate values. Then it is downloaded.

**Run:** CPU is enough, GPU is faster. Run all from top to bottom; the data and checkpoint stay where FFM_3_8 found them.

In [ ]:
!pip install healpy -q


In [ ]:
# ==========================================================================
# Cell 0 — paths · data (zip checked by SHA-256, then unpacked to local disk; every array read is checked against the manifest)
# ==========================================================================
import torch
import os, json, time, hashlib, zipfile, shutil
import numpy as np
import healpy as hp
from google.colab import drive, files
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
PATHS = dict(
    data_zip = f'{DRIVE}/ELEC5021/data_48k_2026-10-10.zip',
    ckpt     = f'{SAVE_DIR}/FFM48k_measim_best.pth',
    dev_oof  = f'{SAVE_DIR}/mesh2z_48k_v1/dev_oof.npz',      # written by FFM_3_8 Cell 8
)
for k, p in PATHS.items():
    print(f'{k:9s} {"OK " if os.path.exists(p) else "MISSING"} {p}')
assert all(os.path.exists(p) for p in PATHS.values())
ZIP_SHA256 = 'f51c6b072123e70d6d8a0691373ab90e3f6db9856ce6306f65d949c87fbf97c7'
SPLIT_SHA256 = 'dc89bc454fef351951283cccb3772fdd3e552bbdde5d13668f8f576f4cd83561'   # only for the checkpoint assert
OUT = f'{SAVE_DIR}/pipeline_fig'
os.makedirs(OUT, exist_ok=True)


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def unzip_checked(zip_path, sha, dest):
    stamp = f'{dest}/.zip_ok_{sha[:16]}'
    if os.path.exists(stamp):
        return
    t0 = time.time()
    assert sha256(zip_path) == sha, f'{zip_path}: SHA-256 mismatch'
    shutil.rmtree(dest, ignore_errors=True); os.makedirs(dest)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    open(stamp, 'w').write(sha)
    print(f'{os.path.basename(zip_path)} checked and unpacked ({time.time() - t0:.0f}s)')


unzip_checked(PATHS['data_zip'], ZIP_SHA256, '/content/data_48k_local')
DATA = '/content/data_48k_local/data_48k'
MIRROR = np.load(f'{DATA}/healpix_mirror_idx.npy')
assert np.array_equal(MIRROR[MIRROR], np.arange(3072))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_grad_enabled(False)
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# ==========================================================================
# Cell 1 — settings (same as FFM_3_8 Cell 1)
# ==========================================================================
N_BINS, FS = 86, 48000
FREQS = np.arange(N_BINS) * FS / 256
EVAL_BINS = slice(1, N_BINS)              # bins 1-85 = 187.5 Hz - 15.94 kHz, DC excluded
EL = 90 - np.degrees(hp.pix2ang(16, np.arange(3072))[0])
MEAS_DIRS = EL >= -45.0                   # 2652 directions
DM = 2                                    # FiLM row of FFM48k_measim for SONICOM measurements
DS = 4                                    # FiLM row for SONICOM simulations (only for the checkpoint assert)
SUBJECT = 'SON_P0010'


In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（GELU + FiLM）
#
# 与 FFM48k_Train.ipynb 的模型定义逐字相同（构建时复制进来）。
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 载入 FFM48k_measim
# ==========================================================================
ck = torch.load(PATHS['ckpt'], map_location='cpu', weights_only=False)
cfg = ck['config']
assert ck['run_name'] == 'FFM48k_measim' and cfg['num_bins'] == N_BINS and cfg['num_datasets'] == 6
assert cfg['data_zip_sha256'] == ZIP_SHA256 and cfg['split_sha256'] == SPLIT_SHA256
model = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=64, freq_embed_dim=32, num_datasets=6,
                             num_bins=N_BINS, tie_encoder_film=True).to(device)
model.load_state_dict(ck['model_state_dict']); model.eval()
FILM_ID = {d['tag']: d['id'] for d in ck['datasets']}
assert FILM_ID['SONICOM'] == DM and FILM_ID['SONICOM_SIM'] == DS
print(f"FFM48k_measim epoch {ck['epoch'] + 1} | val {ck['val_rmse']:.4f} dB | FiLM 行 {FILM_ID}")
del ck

In [ ]:
# ==========================================================================
# Cell 4 — P0010: measured, reconstruction, z, predicted z (out of fold), decoded prediction, fold mean · gate · save
# ==========================================================================
def load_rows(tag, ids):
    # 每人两行（左耳, 镜像后的右耳）；每个数组按 manifest 核对 SHA-256
    d = f'{DATA}/{tag}_48k'
    man = {m['id']: m for m in json.load(open(f'{d}/manifest.json'))}
    X = np.empty((2 * len(ids), N_BINS, 3072), np.float32)
    for k, s in enumerate(ids):
        for side in ('left', 'right'):
            f = f'{d}/npy_{side}/{man[s][side]}'
            assert sha256(f) == man[s][f'{side}_sha256'], f
            a = np.load(f)[:N_BINS]
            X[2 * k + (side == 'right')] = a[:, MIRROR] if side == 'right' else a
    return X


O = np.load(PATHS['dev_oof'])
DEV = [str(s) for s in O['dev']]; fold = O['fold']
assert len(DEV) == 164 and SUBJECT in DEV
k = DEV.index(SUBJECT)
print(f'{SUBJECT}: development index {k}, fold {fold[k]}')

X = load_rows('SONICOM', DEV)                                  # (328, 86, 3072): left, mirrored right
rows = [2 * k, 2 * k + 1]
tr = np.flatnonzero(np.repeat(fold, 2) != fold[k])             # training rows of P0010's fold
MU = X[tr].astype(np.float64).mean(0).astype(np.float32)

did = lambda n: torch.full((n,), DM, dtype=torch.long, device=device)
x = torch.from_numpy(X[rows]).to(device)
rec, z = model(x, did(2))
zhat = torch.from_numpy(O['zpred_geometry'][rows].reshape(2, N_BINS, 64)).float().to(device)
pred = model.decoder(zhat, did(2))
rec, z, pred = rec.float().cpu().numpy(), z.float().cpu().numpy(), pred.float().cpu().numpy()
zhat = zhat.cpu().numpy()


def lsd_el45(p, g):
    e = (p - g)[EVAL_BINS]
    return float(np.sqrt((e ** 2).mean(0))[MEAS_DIRS].mean())


got = dict(geometry=np.mean([lsd_el45(pred[e], X[rows[e]]) for e in (0, 1)]),
           mean=np.mean([lsd_el45(MU, X[rows[e]]) for e in (0, 1)]),
           reconstruction=np.mean([lsd_el45(rec[e], X[rows[e]]) for e in (0, 1)]))
want = dict(geometry=float(O['subject_geometry_lsd_el45'][k]), mean=float(O['subject_mean_lsd_el45'][k]))
for key in want:
    print(f'gate {key:9s} recorded {want[key]:.4f}  now {got[key]:.4f}')
    assert abs(got[key] - want[key]) < 1e-3, f'gate failed for {key}: row order, FiLM row or fold is wrong'
print(f"AE reconstruction LSD (both ears, not gated): {got['reconstruction']:.4f} dB")

np.savez_compressed(f'{OUT}/pipeline_P0010.npz', subject=SUBJECT, freqs=FREQS,
                    measured=X[rows[0]], reconstructed=rec[0], predicted=pred[0], mean=MU,
                    z=z[0], zhat=zhat[0], lsd_geometry=got['geometry'], lsd_mean=got['mean'],
                    lsd_reconstruction=got['reconstruction'], fold=int(fold[k]))
print('->', f'{OUT}/pipeline_P0010.npz  (left ear, own frame; arrays in dB, latents 86 x 64)')
files.download(f'{OUT}/pipeline_P0010.npz')


## After running

Put `pipeline_P0010.npz` in `hrtf-GitHub/results/pipeline_fig/` and tell Claude. The panels (HEALPix maps without the −45° line, latent images, colour bar) are drawn locally from this file.